In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp
import ROOT
from ROOT import TMVA

import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec
from analysis_village.cc1pi.CutMasks import CutMasks

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

from analysis_village.cc1pi.dEdxCleaning import dEdxCleaning

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

np.seterr(divide='ignore', invalid='ignore', over='ignore')
# Import the constants
import analysis_village.cc1pi.makedf.make_cc1pidf as make_cc1pidf

# Load DataFrames

In [ ]:
## Check keys in each file
test_file = "../../../test_data/cc1pi_test.df"
print("keys in test_file")
splh.print_keys(test_file)

## Check split multiplicity
print("mc_bnb_cosmic_file n_split: %d" %splh.get_n_split(test_file))

In [ ]:
## Define keys to load
print('MC dataframes')
n_max_concat = 2 ## for big files, each key could have more than one split
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf","hit0","hit1","hit2"] ## keys from the configuration file
test_df = splh.load_dfs(test_file, keys2load, n_max_concat)
print('test data loaded!')

In [ ]:
evt_df = test_df['cc1pi']
hdr_df = test_df['hdr']
nu_df = test_df['nudf']
hit0_df = test_df['hit0']
hit1_df = test_df['hit1']
hit2_df = test_df['hit2']
hit_dfs = [hit0_df, hit1_df, hit2_df]
best_hit_df = make_cc1pidf.get_best_hit_df(evt_df, hit_dfs)

# Do truth matching

# Test background composition

In [ ]:
slc_df = evt_df
obvious_cosmic_mask = slc_df.slc.cut.obvious_cosmic == True
t0_mask = slc_df.slc.cut.t0
is_inside_FV_mask = slc_df.slc.cut.inside_FV == True
nu_score_mask = slc_df.slc.cut.nu_score == True
MIP_df = slc_df[obvious_cosmic_mask & t0_mask & is_inside_FV_mask & nu_score_mask & ~CutMasks.exiting_pfp_mask(slc_df) & CutMasks.is_MIP_candidate_mask(slc_df) ]

In [ ]:
group_levels = [
    "__ntuple",
    "entry",
    "rec.slc..index",
    "rec.slc.reco.pfp..index"
]

fixed_hit_df = (
    best_hit_df
    .groupby(level=group_levels, group_keys=False)
    .apply(dEdxCleaning.get_fix_hit_df, plot = True)
)


In [ ]:
fixed_hit_df